In [1]:
pip install datasets transformers


Note: you may need to restart the kernel to use updated packages.


In [2]:
from datasets import load_dataset

# Load the CNN/DailyMail dataset
dataset = load_dataset("cnn_dailymail", "3.0.0")

# View the keys
print(dataset.keys())  # Should show train, validation, test

# Preview a sample
sample = dataset['train'][0]
print("Article:\n", sample['article'])
print("\nSummary:\n", sample['highlights'])


dict_keys(['train', 'validation', 'test'])
Article:
 LONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains access to a reported £20 million ($41.1 million) fortune as he turns 18 on Monday, but he insists the money won't cast a spell on him. Daniel Radcliffe as Harry Potter in "Harry Potter and the Order of the Phoenix" To the disappointment of gossip columnists around the world, the young actor says he has no plans to fritter his cash away on fast cars, drink and celebrity parties. "I don't plan to be one of those people who, as soon as they turn 18, suddenly buy themselves a massive sports car collection or something similar," he told an Australian interviewer earlier this month. "I don't think I'll be particularly extravagant. "The things I like buying are things that cost about 10 pounds -- books and CDs and DVDs." At 18, Radcliffe will be able to gamble in a casino, buy a drink in a pub or see the horror film "Hostel: Part II," currently six places below his number

🔹 Quick Notes:
The "article" field contains the news article.

The "highlights" field contains the human-written summary.

You can access subsets like dataset['train'], dataset['validation'], and dataset['test'].


⚠️ What Are These Warnings?
1. HBoxModel Jupyter Widgets Warning
This is related to a UI widget not loading properly in JupyterLab.

Doesn't affect dataset loading or model training.

We can ignore this if we're not using widgets like sliders or progress bars.

2. huggingface_hub symlink warning
Windows has limited support for symbolic links unless:

We run as Administrator

OR enable Developer Mode on Windows

👉 The dataset is still downloaded normally, just with some duplication on your disk.

💡 We can ignore this warning

✅ Step 2: Load a Pretrained Summarization Model
We'll use Hugging Face's transformers library and load a model like facebook/bart-large-cnn—a popular model fine-tuned specifically for summarization tasks.

In [3]:
!pip install transformers
!pip install sentencepiece


🔄 Step 3: Summarize a Sample Article

In [4]:
!pip install torch


In [5]:
pip install --upgrade typing_extensions


Note: you may need to restart the kernel to use updated packages.


In [6]:
import torch
print(torch.__version__)


2.2.2+cpu


In [7]:
import torch
print(torch.__version__)  # This will print the installed version of PyTorch


2.2.2+cpu


In [8]:
pip install torch==2.2.2


Note: you may need to restart the kernel to use updated packages.


In [9]:
from transformers import pipeline, BartTokenizer, BartForConditionalGeneration

# Load model and tokenizer explicitly
model = BartForConditionalGeneration.from_pretrained("facebook/bart-large-cnn")
tokenizer = BartTokenizer.from_pretrained("facebook/bart-large-cnn")

# Create the summarizer pipeline with explicit model and tokenizer
summarizer = pipeline("summarization", model=model, tokenizer=tokenizer, framework="pt")

# Example text to summarize
text = "The CNN/Daily Mail dataset is commonly used for text summarization tasks in NLP. It contains news articles and their human-written summaries."

# Get the summary
summary = summarizer(text, max_length=130, min_length=30, do_sample=False)
print(summary[0]['summary_text'])


model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

C:\Users\PMYLS\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:144: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\PMYLS\.cache\huggingface\hub\models--facebook--bart-large-cnn. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Device set to use cpu
Your max_length is set to 130, but your input_length is only 32. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=16)


The CNN/Daily Mail dataset is commonly used for text summarization tasks in NLP. It contains news articles and their human-written summaries.


📝 Notes:
max_length and min_length define the bounds of the summary length.

do_sample=False makes the output deterministic (same output every time).



🔹 Step 3: Load a Pretrained Summarization Model (BART)
We'll use facebook/bart-large-cnn as it's already fine-tuned on CNN/Daily Mail.

In [11]:
from transformers import pipeline

# Force the use of PyTorch instead of TensorFlow
summarizer = pipeline("summarization", model="facebook/bart-large-cnn", framework="pt")


Device set to use cpu


In [14]:
from transformers import pipeline

# Use PyTorch only
summarizer = pipeline("summarization", model="facebook/bart-large-cnn", framework="pt")


Device set to use cpu


In [15]:
text = """The Hugging Face Transformers library provides thousands of pre-trained models to perform tasks on texts such as classification, information extraction, question answering, summarization, translation, and more."""
summary = summarizer(text, max_length=50, min_length=20, do_sample=False)

print(summary[0]['summary_text'])


Your max_length is set to 50, but your input_length is only 38. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=19)


The Hugging Face Transformers library provides thousands of pre-trained models to perform tasks on texts such as classification, information extraction, question answering, summarization, translation, and more.


🔹 Step 4: Generate Summaries for the Test Set
We’ll now summarize a small batch (e.g., 10 samples) for demonstration.

In [16]:
# Let's summarize the first 10 articles
num_samples = 10
summaries = []

for i in range(num_samples):
    article = dataset['test'][i]['article']
    reference = dataset['test'][i]['highlights']
    summary = summarizer(article, max_length=130, min_length=30, do_sample=False)[0]['summary_text']
    summaries.append({
        "article": article,
        "reference_summary": reference,
        "generated_summary": summary
    })

# Preview one result
print("Reference Summary:\n", summaries[0]["reference_summary"])
print("\nGenerated Summary:\n", summaries[0]["generated_summary"])


Reference Summary:
 Membership gives the ICC jurisdiction over alleged crimes committed in Palestinian territories since last June .
Israel and the United States opposed the move, which could open the door to war crimes investigations against Israelis .

Generated Summary:
 The Palestinian Authority becomes the 123rd member of the International Criminal Court. The move gives the court jurisdiction over alleged crimes in Palestinian territories. Israel and the United States opposed the Palestinians' efforts to join the body.


In [19]:
text = """
Machine learning is a field of computer science that gives computers the ability to learn without being explicitly programmed. It is under the umbrella of Artificial Intelligence. The world is advancing day by day
Deep learning is a subset of machine learning that uses neural networks with many layers.
"""

summary = summarizer(text, max_length=50, min_length=25, do_sample=False)
print("Summary:", summary[0]['summary_text'])


Summary: Machine learning is a field of computer science that gives computers the ability to learn without being explicitly programmed. It is under the umbrella of Artificial Intelligence.


🔹 Step 5: Evaluate with ROUGE Score

In [20]:
pip install evaluate


   ---------------------------------------- 0.0/84.0 kB ? eta -:--:--
   ---- ----------------------------------- 10.2/84.0 kB ? eta -:--:--
   -------------- ------------------------- 30.7/84.0 kB 435.7 kB/s eta 0:00:01
   ------------------- -------------------- 41.0/84.0 kB 326.8 kB/s eta 0:00:01
   ----------------------------- ---------- 61.4/84.0 kB 469.7 kB/s eta 0:00:01
   ---------------------------------------- 84.0/84.0 kB 472.9 kB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [22]:
!pip install rouge_score


  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24970 sha256=96f5a3d875488bf3c861b5562eac3f887beda823a022d24db802a7a7ebb92804
  Stored in directory: c:\users\pmyls\appdata\local\pip\cache\wheels\1e\19\43\8a442dc83660ca25e163e1bd1f89919284ab0d0c1475475148
Successfully built rouge_score


In [23]:
import evaluate

# Load ROUGE metric
rouge = evaluate.load("rouge")

# Prepare lists for evaluation
reference_summaries = [item['reference_summary'] for item in summaries]
generated_summaries = [item['generated_summary'] for item in summaries]

# Compute ROUGE scores
scores = rouge.compute(predictions=generated_summaries, references=reference_summaries)

# Display scores
print("ROUGE Evaluation:\n")
for key in scores:
    print(f"{key}: {scores[key]:.4f}")


ROUGE Evaluation:

rouge1: 0.4049
rouge2: 0.2040
rougeL: 0.3137
rougeLsum: 0.3478
